# Levishtien Distance Type-1 Embedding Model

"కల"   → ['క','ల']   
"కల్ల" → ['క','ల','్','ల']   
"కాల"  → ['క','ా','ల']

d("కల","కల్ల") = 2   (insert '్', insert 'ల')        
d("కల","కాల")  = 1   (insert 'ా')       
d("కల్ల","కాల") = 2


In [ ]:
words = ["కల","కల్ల","కాల","కళ్ళు"]

In [2]:
chars=[ch for ch in words[3]]

In [3]:
chars

['క', 'ళ', '్', 'ళ', 'ు']

In [4]:
vocab=set()
for word in words:
    for ch in word:
        vocab.add(ch)
vocab

{'క', 'ల', 'ళ', 'ా', 'ు', '్'}

In [5]:
vocab=['<pad>','<unk>']+sorted(vocab)
char2id={ch:i for i,ch in enumerate(vocab)}
id2char={i:ch for i,ch in enumerate(vocab)}
print(vocab)
print(char2id)
print(id2char)

['<pad>', '<unk>', 'క', 'ల', 'ళ', 'ా', 'ు', '్']
{'<pad>': 0, '<unk>': 1, 'క': 2, 'ల': 3, 'ళ': 4, 'ా': 5, 'ు': 6, '్': 7}
{0: '<pad>', 1: '<unk>', 2: 'క', 3: 'ల', 4: 'ళ', 5: 'ా', 6: 'ు', 7: '్'}


In [8]:
def build_vocab(words_list):
    vocab=set()
    for word in words_list:
        for ch in word:
            vocab.add(ch)
    vocab=['<pad>','<unk>']+sorted(vocab)
    char2id={ch:i for i,ch in enumerate(vocab)}
    id2char={i:ch for i,ch in enumerate(vocab)}
    return vocab,char2id,id2char

In [6]:
PAD=char2id['<pad>']
def encode(word,max_len):
    ids=[char2id.get(ch,char2id['<unk>']) for ch in word]
    return ids+[PAD]*(max_len-len(ids))

In [7]:
encode(words[2],10)

[2, 5, 3, 0, 0, 0, 0, 0, 0, 0]

In [9]:
max_len=max(len(w) for w in words)
encode(words[1],max_len)

[2, 3, 7, 3, 0]

In [10]:
def levenshtein(a,b):
    n, m = len(a),len(b)
    dp = [[0]*(m+1) for _ in range(n+1)]

    for i in range(n+1):
        dp[i][0] = i
    for j in range(m+1):
        dp[0][j] = j

    for i in range(1, n+1):
        for j in range(1, m+1):
            cost = 0 if a[i-1] == b[j-1] else 1
            dp[i][j] = min(
                dp[i-1][j] + 1,      # deletion
                dp[i][j-1] + 1,      # insertion
                dp[i-1][j-1] + cost  # substitution
            )
    return dp[n][m]

In [23]:
def levenshtein(a, b):
    n, m = len(a), len(b)
    dp = [[0]*(m+1) for _ in range(n+1)]

    for i in range(n+1):
        dp[i][0] = i
    for j in range(m+1):
        dp[0][j] = j

    for i in range(1, n+1):
        for j in range(1, m+1):
            cost=1
            if a[i-1]==b[j-1]:
                cost=0
            dp[i][j]=min(dp[i-1][j]+1, dp[i][j-1]+1,dp[i-1][j-1]+cost)
    return dp[n][m]


In [24]:
list(words[2])

['క', 'ా', 'ల']

In [25]:
levenshtein(list("కల"), list("కల్ల")) 

2

In [17]:
pairs = []
for i in range(len(words)):
    for j in range(len(words)):
        w1, w2 = words[i], words[j]
        d = levenshtein(list(w1), list(w2))
        pairs.append((w1, w2, d))

In [18]:
pairs

[('కల', 'కల', 0),
 ('కల', 'కల్ల', 2),
 ('కల', 'కాల', 1),
 ('కల', 'కళ్ళు', 4),
 ('కల్ల', 'కల', 2),
 ('కల్ల', 'కల్ల', 0),
 ('కల్ల', 'కాల', 2),
 ('కల్ల', 'కళ్ళు', 3),
 ('కాల', 'కల', 1),
 ('కాల', 'కల్ల', 2),
 ('కాల', 'కాల', 0),
 ('కాల', 'కళ్ళు', 4),
 ('కళ్ళు', 'కల', 4),
 ('కళ్ళు', 'కల్ల', 3),
 ('కళ్ళు', 'కాల', 4),
 ('కళ్ళు', 'కళ్ళు', 0)]

In [21]:
import torch

X1, X2, Y = [], [], []

for w1, w2, d in pairs:
    X1.append(encode(w1,max_len))
    X2.append(encode(w2,max_len))
    Y.append(d)

X1 = torch.tensor(X1)
X2 = torch.tensor(X2)
Y  = torch.tensor(Y, dtype=torch.float32).unsqueeze(1)


# Embedding Siamese Model

In [26]:
PAD

0

In [27]:
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self,vocab_size,emb_dim,hid_dim):
        super().__init__()
        self.emb=nn.Embedding(vocab_size,emb_dim,padding_idx=PAD)
        self.lstm=nn.LSTM(emb_dim,hid_dim,batch_first=True,bidirectional=True)
    
    def forward(self,x):
        x=self.emb(x)
        _,(h,_)=self.lstm(x)
        return torch.cat((h[-2],h[-1]),dim=1)
    

In [29]:
class Siamese(nn.Module):
    def __init__(self, encoder, hid_dim):
        super().__init__()
        self.encoder = encoder
        self.fc = nn.Linear(hid_dim*2,1)

    def forward(self, x1, x2):
        v1 = self.encoder(x1)
        v2 = self.encoder(x2)
        return self.fc(torch.abs(v1-v2))


In [30]:
model = Siamese(Encoder(len(vocab), emb_dim=16, hid_dim=32),hid_dim=32)


In [31]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

for epoch in range(200):
    optimizer.zero_grad()
    preds = model(X1, X2)
    loss = loss_fn(preds, Y)
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"epoch {epoch} | loss {loss.item():.4f}")


epoch 0 | loss 6.8153
epoch 50 | loss 1.3212
epoch 100 | loss 0.1079
epoch 150 | loss 0.0080


In [32]:
with torch.no_grad():
    test1=encode("కల",max_len)
    test2=encode("కల్ల",max_len)
    
    pred=model(torch.tensor([test1]),torch.tensor([test2]))
print("Predicted Distance: ",pred.item())

Predicted Distance:  1.9853217601776123


# Approach-2 (Seq Alignment Model)

In [33]:
import torch
import torch.nn as nn

class SeqAlignModel(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim):
        super().__init__()
        self.emb = nn.Embedding(vocab_size,emb_dim,padding_idx=PAD)
        self.lstm = nn.LSTM(
            emb_dim,
            hid_dim,
            batch_first=True,
            bidirectional=True
        )
        self.fc = nn.Linear(hid_dim*2,1)

    def forward(self, x1, x2):
        e1, _ = self.lstm(self.emb(x1))
        e2, _ = self.lstm(self.emb(x2))

        diff = torch.abs(e1-e2)
        diff = diff.mean(dim=1)

        return self.fc(diff)


In [34]:
model2 = SeqAlignModel(len(vocab), emb_dim=16, hid_dim=32)

optimizer = torch.optim.Adam(model2.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

In [35]:
for epoch in range(200):
    optimizer.zero_grad()
    preds = model2(X1, X2)
    loss = loss_fn(preds, Y)
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"[SeqAlign] epoch {epoch} | loss {loss.item():.4f}")

[SeqAlign] epoch 0 | loss 6.3694
[SeqAlign] epoch 50 | loss 2.6321
[SeqAlign] epoch 100 | loss 0.2811
[SeqAlign] epoch 150 | loss 0.0149


In [39]:
with torch.no_grad():
    test1=encode("కల",max_len)
    test2=encode("కల్ల",max_len)
    
    pred2=model2(torch.tensor([test1]),torch.tensor([test2]))
print("Predicted Distance: ",pred2.item())

Predicted Distance:  1.9305132627487183


# Approach-3 CNN-based Stacking

In [37]:
class CNNSiamese(nn.Module):
    def __init__(self, vocab_size, emb_dim):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        self.conv = nn.Conv2d(
            in_channels=2,
            out_channels=16,
            kernel_size=(3, emb_dim)
        )
        self.fc = nn.Linear(16, 1)

    def forward(self, x1, x2):
        e1 = self.emb(x1).unsqueeze(1)
        e2 = self.emb(x2).unsqueeze(1)

        x = torch.cat([e1, e2], dim=1)

        x = self.conv(x)               
        x = x.squeeze(-1).mean(dim=2)

        return self.fc(x)


In [38]:
model3 = CNNSiamese(len(vocab), emb_dim=16)

optimizer = torch.optim.Adam(model3.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

for epoch in range(200):
    optimizer.zero_grad()
    preds = model3(X1, X2)
    loss = loss_fn(preds, Y)
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"[CNN] epoch {epoch} | loss {loss.item():.4f}")


[CNN] epoch 0 | loss 6.6358
[CNN] epoch 50 | loss 1.9909
[CNN] epoch 100 | loss 1.9083
[CNN] epoch 150 | loss 1.8888


In [40]:
with torch.no_grad():
    test1=encode("కల",max_len)
    test2=encode("కల్ల",max_len)
    
    pred3=model3(torch.tensor([test1]),torch.tensor([test2]))
print("Predicted Distance: ",pred3.item())

Predicted Distance:  1.4667778015136719
